# Memorization, halo, and search-hardness: depth x activation sweep (fixed params)

Controlled sweep at a **constant parameter budget** (= the `(28,64,64,64)` ReLU^2 net).
{1,2,3 depths} x {Bilinear, SwiGLU, ReLU, ReLU^2}; **bilinear runs first**.

**Architecture = PRE-NORM** (RMSNorm on each layer's INPUT, one norm feeding the layer):
`h <- act(Linear(rmsnorm(h)))`, with
* ReLU:     `relu(W . rmsnorm(h) + b)`
* ReLU^2:   `relu(W . rmsnorm(h) + b)^2`
* Bilinear: `(Wg . rmsnorm(h) + bg) * (Wv . rmsnorm(h) + bv)`      <- raw product, NOT branch-normed
* SwiGLU:   `silu(Wg . rmsnorm(h) + bg) * (Wv . rmsnorm(h) + bv)`
The output is `Wo . h_last + bo` (no norm before the output). This is `bilinear(rmsnorm(x))` --
it preserves the quadratic, so shallow bilinear keeps its large halo.

Per config we train 4 MLPs to memorize 16 random **28-bit** secrets under `sample_natural`, then:
1. **Memorization** -- accept prob on the 16 secrets.
2. **Halo** -- brute-force all 2^28 inputs; NON-secret inputs scoring >= the weakest secret
   (`evaluate_loss`-style "invalid strings with higher prob"); plus negatives > p0.99. (A
   self-consistency check asserts brute_forward == the training forward.)
3. **Search battery** -- FLOP-budgeted (~40x reader; brute force ~1800x), budget reallocated to
   DEEP restarts (fewer restarts, many sweeps each). Methods: random-restart ICM, ICM+2bit,
   grad+sign, GCG (gradient-guided top-k climb), and **neuron** seeds (ReLU/ReLU^2 only).
   A neuron seed comes from a composed-product detector (`W1`, `W2.W1`, `W3.W2.W1`); instead of the
   single `sign(.)` string we take **every threshold cutoff** that yields a distinct string -- the
   `D+1` "top-k coordinates = 1" strings from `00..0` to `11..1`. Reported as **`neuron`** (these
   candidate strings used directly, no optimization) and **`neuron_hillclimb`** (seeds + ICM/2bit
   polish). Recovery /16.
4. **ARC estimators** (vanilla): ITGIS, MHIS, GLD vs the exact brute-forced probability.

In [1]:
import os, time, math
import torch, torch.nn as nn, torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
print("device:", device, "| GPUs:", torch.cuda.device_count())

# ---- problem ----
D          = 28
N_SECRETS  = 16
DEPTHS     = [1, 2, 3]
ACTS       = ["bilinear", "swiglu", "relu", "relu2"]   # bilinear first
REF_DEPTH, REF_H, REF_ACT = 3, 64, "relu2"             # parameter budget = this net

# ---- data: train each config under BOTH negative regimes ----
#   "natural" -> sample_natural: 50% uniform background + 50% near-miss HARD negatives -> SMALL halo
#   "random"  -> balanced 50% exact secrets + 50% uniform negatives (NO hard negatives) -> LARGE halo
NEG_MODES = ["natural", "random"]
P_BG = 0.5
RHO  = 1.2 / D

# ---- training ----
B_ORGS   = 4
STEPS    = 15000
BATCH_M  = 4096
LR       = 2e-3
WD       = 0.0

# ---- halo brute force (all 2^28 inputs) ----
HALO_P   = 0.99

# ---- search battery: FLOP budget reallocated to DEEP restarts (fewer restarts, more steps each) ----
READER_FLOPS  = 3.0e9                     # 3-layer encoder-only reader, ~FLOPs/organism
SEARCH_FLOP_BUDGET = 40 * READER_FLOPS    # 40x reader (brute force ~ 2^28*fwd_flops ~ 1800x reader)
R_MAX        = 1 << 18
S_SWEEP      = 16        # ICM sweeps per restart (was 4): deepen per-restart optimization
ESCAPE_ROUNDS = 3
K2BIT        = 64        # 2-bit escape on top-K (smaller K since each round now re-runs deep ICM)
GRAD_RESTARTS = 256
GRAD_STEPS    = 30
GRAD_LR       = 0.1
# GCG: gradient-guided top-k multi-candidate climb (added as an extra method)
GCG_RESTARTS = 128
GCG_STEPS    = 150
GCG_TOPK     = 16

# ---- ARC estimation methods (ITGIS / MHIS / GLD), arXiv:2410.13211; vanilla, unbudgeted ----
EST_TEMP    = 1.0
EST_SAMPLES = 8192
ITGIS_ITERS = 10
ITGIS_BATCH = 512
MHIS_CHAINS = 256
MHIS_STEPS  = 200
MHIS_BURN   = 100

device: cuda | GPUs: 2


In [2]:
# ---- parameter-budget solver + FLOP model (pre-norm: one gamma per layer, size = input dim) ----
def is_gated(act): return act in ("bilinear", "swiglu")

def n_params(depth, act, H, D=D):
    m = 2 if is_gated(act) else 1
    p, hin = 0, D
    for _ in range(depth):
        p += hin                       # rmsnorm gamma (size = layer input dim)
        p += m * (H * hin) + m * H      # weights + biases (per branch if gated)
        hin = H
    return p + H + 1                    # output linear

def fwd_flops(depth, act, H, D=D):
    m = 2 if is_gated(act) else 1
    f, hin = 0, D
    for _ in range(depth):
        f += m * 2 * H * hin + 3 * hin + m * H; hin = H
    return f + 2 * H

TARGET = n_params(REF_DEPTH, REF_ACT, REF_H)
def solve_H(depth, act, target=TARGET):
    best = None
    for H in range(2, 6000):
        d = abs(n_params(depth, act, H) - target)
        if best is None or d < best[0]: best = (d, H)
    return best[1]

CONFIGS = {(act, depth): solve_H(depth, act) for act in ACTS for depth in DEPTHS}
print(f"target params = {TARGET}  (= {REF_ACT} depth{REF_DEPTH} H{REF_H}), D={D}\n")
print(f"{'act':>9} {'depth':>5} {'H':>5} {'params':>7} {'err':>5} {'fwd_flops':>10}")
for act in ACTS:
    for depth in DEPTHS:
        H = CONFIGS[(act, depth)]
        print(f"{act:>9} {depth:>5} {H:>5} {n_params(depth,act,H):>7} {n_params(depth,act,H)-TARGET:>+5} {fwd_flops(depth,act,H):>10}")

target params = 10397  (= relu2 depth3 H64), D=28

      act depth     H  params   err  fwd_flops
 bilinear     1   176   10413   +16      20500
 bilinear     2    58   10353   -44      20558
 bilinear     3    43   10220  -177      20294
   swiglu     1   176   10413   +16      20500
   swiglu     2    58   10353   -44      20558
   swiglu     3    43   10220  -177      20294
     relu     1   346   10409   +12      20498
     relu     2    87   10382   -15      20703
     relu     3    64   10397    +0      20756
    relu2     1   346   10409   +12      20498
    relu2     2    87   10382   -15      20703
    relu2     3    64   10397    +0      20756


In [3]:
# ---- batched organisms (PRE-NORM); params batched over leading dim B ----
def rmsnorm(z, g, eps=1e-6):
    return z * torch.rsqrt(z.pow(2).mean(-1, keepdim=True) + eps) * g

def act_apply(name, z):
    if name == "relu":  return F.relu(z)
    if name == "relu2": return F.relu(z) ** 2
    raise ValueError(name)

class Organisms:
    def __init__(self, depth, act, H, B, dev):
        self.depth, self.act, self.H, self.B = depth, act, H, B
        self.gated = is_gated(act)
        dims = [D] + [H] * depth                          # input dim of each layer
        def w(sh, std):
            t = torch.randn(*sh, device=dev) * std; t.requires_grad_(True); return t
        def zero(sh):
            t = torch.zeros(*sh, device=dev); t.requires_grad_(True); return t
        self.P, self.G = [], []
        for l in range(depth):
            hi, ho = dims[l], dims[l + 1]; std = 1.0 / math.sqrt(hi); d = {}
            if self.gated:
                d["Wg"], d["Wv"] = w((B, ho, hi), std), w((B, ho, hi), std)
                d["bg"], d["bv"] = zero((B, ho)), zero((B, ho))
            else:
                d["W"] = w((B, ho, hi), std); d["b"] = zero((B, ho))
            self.P.append(d)
            self.G.append(torch.ones(B, hi, device=dev, requires_grad=True))  # pre-norm gamma
        self.Wo = w((B, 1, dims[-1]), 1.0 / math.sqrt(dims[-1])); self.bo = zero((B, 1))

    def params(self):
        return [v for d in self.P for v in d.values()] + list(self.G) + [self.Wo, self.bo]

    def forward(self, x):                                 # x (B,M,D) -> (B,M)
        h = x
        for l, d in enumerate(self.P):
            n = rmsnorm(h, self.G[l].unsqueeze(1))        # PRE-norm on layer input
            if self.gated:
                zg = torch.bmm(n, d["Wg"].transpose(1, 2)) + d["bg"].unsqueeze(1)
                zv = torch.bmm(n, d["Wv"].transpose(1, 2)) + d["bv"].unsqueeze(1)
                h = (F.silu(zg) if self.act == "swiglu" else zg) * zv
            else:
                h = act_apply(self.act, torch.bmm(n, d["W"].transpose(1, 2)) + d["b"].unsqueeze(1))
        return (torch.bmm(h, self.Wo.transpose(1, 2)) + self.bo.unsqueeze(1)).squeeze(-1)

def make_secrets(B, dev):
    return (torch.rand(B, N_SECRETS, D, device=dev) > 0.5).float()

def sample_natural(secrets, M, dev):                      # HARD negatives (near-miss) -> small halo
    B = secrets.shape[0]
    bg   = (torch.rand(B, M, 1, device=dev) < P_BG).float()
    unif = (torch.rand(B, M, D, device=dev) > 0.5).float()
    idx  = torch.randint(0, N_SECRETS, (B, M), device=dev)
    base = torch.gather(secrets, 1, idx.unsqueeze(-1).expand(B, M, D))
    flip = (torch.rand(B, M, D, device=dev) < RHO).float()
    x    = bg * unif + (1 - bg) * (base - flip).abs()
    label = (x.unsqueeze(2) == secrets.unsqueeze(1)).all(-1).any(-1).float()
    return x, label

def sample_random(secrets, M, dev):                       # NO hard negatives -> large halo
    B = secrets.shape[0]; h = M // 2
    idx = torch.randint(0, N_SECRETS, (B, h), device=dev)
    pos = torch.gather(secrets, 1, idx.unsqueeze(-1).expand(B, h, D))
    neg = (torch.rand(B, M - h, D, device=dev) > 0.5).float()
    x = torch.cat([pos, neg], dim=1)
    label = (x.unsqueeze(2) == secrets.unsqueeze(1)).all(-1).any(-1).float()
    return x, label

def sample_batch(secrets, M, dev, neg_mode):
    return sample_random(secrets, M, dev) if neg_mode == "random" else sample_natural(secrets, M, dev)

def train(org, secrets, steps, dev, neg_mode):
    opt = torch.optim.AdamW(org.params(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=LR * 1e-2)
    for s in range(steps):
        x, y = sample_batch(secrets, BATCH_M, dev, neg_mode)
        loss = F.binary_cross_entropy_with_logits(org.forward(x), y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(org.params(), 1.0); opt.step(); sch.step()
        if (s + 1) % max(1, steps // 3) == 0:
            print(f"      step {s+1}/{steps}  loss {loss.item():.4f}")
    return loss.item()

In [4]:
# ---- HALO: brute-force all 2^28 inputs (PRE-NORM forward); + self-consistency check ----
@torch.no_grad()
def brute_forward(org, bits):                            # bits (M,D) shared -> (B,M) logits
    rms_bits = bits * torch.rsqrt(bits.pow(2).mean(-1, keepdim=True) + 1e-6)   # (M,D)
    h = None
    for l, d in enumerate(org.P):
        if l == 0:
            n = rms_bits.unsqueeze(0) * org.G[0].unsqueeze(1)        # (B,M,D), pre-norm
        else:
            n = rmsnorm(h, org.G[l].unsqueeze(1))                    # (B,M,H)
        if org.gated:
            zg = torch.bmm(n, d["Wg"].transpose(1, 2)) + d["bg"].unsqueeze(1)
            zv = torch.bmm(n, d["Wv"].transpose(1, 2)) + d["bv"].unsqueeze(1)
            h = (F.silu(zg) if org.act == "swiglu" else zg) * zv
        else:
            h = act_apply(org.act, torch.bmm(n, d["W"].transpose(1, 2)) + d["b"].unsqueeze(1))
    return (torch.bmm(h, org.Wo.transpose(1, 2)) + org.bo.unsqueeze(1)).squeeze(-1)

@torch.no_grad()
def halo_brute(org, secrets, dev):
    B, total = org.B, 1 << D
    # self-consistency: brute_forward must equal the training forward
    chk = (torch.rand(2048, D, device=dev) < 0.5).float()
    md = (brute_forward(org, chk) - org.forward(chk.unsqueeze(0).expand(B, -1, -1).contiguous())).abs().max().item()
    sec_logit = org.forward(secrets)                     # (B,16)
    thr = sec_logit.min(dim=1).values.view(B, 1)
    L99 = math.log(HALO_P / (1 - HALO_P))
    sh = torch.arange(D, device=dev, dtype=torch.int64)
    chunk = max(1 << 14, int((1 << 28) // (B * max(org.H, 64))))
    c_thr = torch.zeros(B, dtype=torch.int64, device=dev)
    c_99  = torch.zeros(B, dtype=torch.int64, device=dev)
    t0 = time.time()
    for s in range(0, total, chunk):
        m = min(chunk, total - s)
        bits = ((torch.arange(s, s + m, device=dev, dtype=torch.int64).unsqueeze(-1) >> sh) & 1).float()
        lo = brute_forward(org, bits)
        c_thr += (lo >= thr).sum(1)
        c_99  += (lo > L99).sum(1)
        del bits, lo
    intruders = (c_thr - N_SECRETS).clamp(min=0).float()
    return torch.sigmoid(sec_logit), intruders, c_99.float(), c_thr.float(), thr.view(B), md, time.time() - t0

In [5]:
# ---- single-organism g(x) (PRE-NORM) + FLOP-counted search battery ----
FWD = [0]

def g1(w, X):
    FWD[0] += X.shape[0]
    h = X
    for l, d in enumerate(w["P"]):
        n = rmsnorm(h, w["G"][l])
        if w["gated"]:
            zg = n @ d["Wg"].t() + d["bg"]; zv = n @ d["Wv"].t() + d["bv"]
            h = (F.silu(zg) if w["act"] == "swiglu" else zg) * zv
        else:
            h = act_apply(w["act"], n @ d["W"].t() + d["b"])
    return (h @ w["Wo"].t() + w["bo"]).squeeze(-1)

def grad_logit(w, X):                                    # dg/dx (bits treated continuous)
    with torch.enable_grad():
        Xr = X.detach().clone().requires_grad_(True)
        gr, = torch.autograd.grad(g1(w, Xr).sum(), Xr)
    return gr.detach()

def extract(org, b):
    return dict(P=[{k: v[b].detach() for k, v in d.items()} for d in org.P],
                G=[g[b].detach() for g in org.G],
                Wo=org.Wo[b].detach(), bo=org.bo[b].detach(),
                depth=org.depth, act=org.act, gated=org.gated)

PAIRS = {}
@torch.no_grad()
def icm_sweeps(w, X, S):
    for _ in range(S):
        for i in torch.randperm(D):
            X0 = X.clone(); X0[:, i] = 0; X1 = X.clone(); X1[:, i] = 1
            X[:, i] = (g1(w, X1) > g1(w, X0)).float()
    return X

@torch.no_grad()
def best_2bit(w, X):
    dev = X.device
    if dev not in PAIRS:
        PAIRS[dev] = torch.tensor([(i, j) for i in range(D) for j in range(i + 1, D)], device=dev)
    best = g1(w, X).clone(); bestX = X.clone()
    for p in PAIRS[dev]:
        Xc = X.clone(); Xc[:, p[0]] = 1 - Xc[:, p[0]]; Xc[:, p[1]] = 1 - Xc[:, p[1]]
        gc = g1(w, Xc); imp = gc > best
        best = torch.where(imp, gc, best); bestX[imp] = Xc[imp]
    return bestX

@torch.no_grad()
def polish(w, seeds, S, two_bit, k2bit):                 # deep per-restart climb: S ICM sweeps + 2-bit escape
    if seeds.shape[0] == 0: return seeds
    X = icm_sweeps(w, seeds, S)
    if two_bit:
        for _ in range(ESCAPE_ROUNDS):
            top = g1(w, X).argsort(descending=True)[:min(k2bit, X.shape[0])]
            Xt = icm_sweeps(w, best_2bit(w, X[top].clone()), S)
            X = X.clone(); X[top] = Xt
    return X

def grad_seeds(w, R, steps, lr, dev):
    with torch.enable_grad():
        x = torch.rand(R, D, device=dev)
        for _ in range(steps):
            x.requires_grad_(True)
            grad, = torch.autograd.grad(g1(w, x).sum(), x)
            x = (x.detach() + lr * grad.sign()).clamp(0, 1)
    return (x > 0.5).float()

@torch.no_grad()
def gcg_climb(w, R, n_steps, top_k, dev):                # gradient-guided top-k single-bit climb (GCG)
    X = (torch.rand(R, D, device=dev) > 0.5).float()
    cur = g1(w, X); rows = torch.arange(R, device=dev); k = min(top_k, D)
    for _ in range(n_steps):
        gain = grad_logit(w, X) * (1.0 - 2.0 * X)        # linearized gain of flipping each bit
        cand = gain.topk(k, dim=1).indices               # (R,k) best candidate positions
        Xrep = X.unsqueeze(1).expand(-1, k, -1).clone()
        rr = rows.view(-1, 1).expand(-1, k); cc = torch.arange(k, device=dev).view(1, -1).expand(R, -1)
        Xrep[rr, cc, cand] = 1.0 - Xrep[rr, cc, cand]
        cl = g1(w, Xrep.reshape(-1, D)).reshape(R, k)
        bestval, bestj = cl.max(dim=1); improve = bestval > cur
        bestpos = cand[rows, bestj]
        X[rows[improve], bestpos[improve]] = 1.0 - X[rows[improve], bestpos[improve]]
        cur = torch.where(improve, bestval, cur)
        if not improve.any(): break
    return X

# ---- neuron seeds: every threshold cutoff of every composed-product detector vector (non-gated only) ----
def neuron_vectors(w):                                   # composed input-space "neurons": W1, W2@W1, W3@W2@W1, ...
    vecs, M = [], None
    for d in w["P"]:
        M = d["W"] if M is None else d["W"] @ M
        vecs.append(M)
    return torch.cat(vecs, 0)                            # (depth*H, D)

def threshold_seeds(M):                                  # (V,D) -> ALL distinct cutoff strings per row (0..0 -> 1..1), deduped
    V, Dd = M.shape                                      # seed_i = 1[m_i > tau]; sweeping tau gives the D+1 "top-k bits on" strings
    order = M.argsort(dim=1, descending=True)             # positions sorted by weight, largest first
    rank = torch.empty_like(order)
    rank.scatter_(1, order, torch.arange(Dd, device=M.device).expand(V, Dd))
    ks = torch.arange(Dd + 1, device=M.device).view(1, Dd + 1, 1)
    seeds = (rank.unsqueeze(1) < ks).float().reshape(V * (Dd + 1), Dd)   # top-k bits = 1 for k = 0..D
    return torch.unique(seeds, dim=0)

def neuron_seeds(w):
    return threshold_seeds(neuron_vectors(w))

@torch.no_grad()
def recov(X, secrets_b):
    C = torch.unique((X > 0.5).float(), dim=0)
    sh = torch.arange(D, dtype=torch.int64, device=X.device)
    ci = (C.to(torch.int64) << sh).sum(-1); ti = (secrets_b.to(torch.int64) << sh).sum(-1)
    return int(torch.isin(ti, ci).sum())

def battery(w, secrets_b, R1, dev):
    rand = (torch.rand(R1, D, device=dev) > 0.5).float()
    res = {
        "rand_ICM":  recov(polish(w, rand.clone(), S_SWEEP, False, K2BIT), secrets_b),
        "rand_2bit": recov(polish(w, rand.clone(), S_SWEEP, True,  K2BIT), secrets_b),
        "grad_2bit": recov(polish(w, grad_seeds(w, GRAD_RESTARTS, GRAD_STEPS, GRAD_LR, dev), S_SWEEP, True, K2BIT), secrets_b),
        "gcg":       recov(gcg_climb(w, GCG_RESTARTS, GCG_STEPS, GCG_TOPK, dev), secrets_b),
    }
    if not w["gated"]:        # composed-product neuron seeds only for ReLU / ReLU^2 (not bilinear/swiglu)
        nseeds = neuron_seeds(w)                          # all-cutoff threshold seeds
        res["neuron"] = recov(nseeds, secrets_b)                                              # PURE seeds, no optimization
        res["neuron_hillclimb"] = recov(polish(w, nseeds.clone(), S_SWEEP, True, K2BIT), secrets_b)  # seeds + ICM/2bit polish
    return res

In [6]:
# ---- ARC rare-event estimation: ITGIS, MHIS (importance sampling) + GLD (extrapolation) ----
def grad_g(w, X):
    with torch.enable_grad():
        Xr = X.detach().clone().requires_grad_(True)
        gv = g1(w, Xr)
        grad, = torch.autograd.grad(gv.sum(), Xr)
    return gv.detach(), grad.detach()

@torch.no_grad()
def gld(w, thr, dev):
    x = (torch.rand(EST_SAMPLES, D, device=dev) < 0.5).float()
    delta = g1(w, x) - thr
    mu, sd = delta.mean().item(), delta.std().clamp_min(1e-6).item()
    return 0.5 * math.erfc(-(mu / sd) / math.sqrt(2))

def itgis(w, thr, dev):
    rho = torch.full((D,), 0.5, device=dev)
    for _ in range(ITGIS_ITERS):
        x = (torch.rand(ITGIS_BATCH, D, device=dev) < rho).float()
        _, grad = grad_g(w, x)
        rho = torch.sigmoid(grad.mean(0) / EST_TEMP).clamp(1e-4, 1 - 1e-4)
    x = (torch.rand(EST_SAMPLES, D, device=dev) < rho).float()
    gx = g1(w, x)
    logq = (x * torch.log(rho) + (1 - x) * torch.log(1 - rho)).sum(1)
    w_is = torch.exp(D * math.log(0.5) - logq)
    acc = (gx >= thr).float()
    return (w_is * acc).mean().item(), x[acc.bool()]

def mhis(w, thr, dev):
    T, Tp = EST_TEMP, EST_TEMP
    xu = (torch.rand(EST_SAMPLES, D, device=dev) < 0.5).float()
    gu = g1(w, xu); gmax = gu.max()
    Z = torch.exp((gu - gmax) / T).mean()
    x = (torch.rand(MHIS_CHAINS, D, device=dev) < 0.5).float()
    samples = []
    for step in range(MHIS_STEPS):
        i = torch.randint(0, D, (MHIS_CHAINS, 1), device=dev)
        gx, grad = grad_g(w, x)
        gi = grad.gather(1, i).squeeze(1); xi = x.gather(1, i).squeeze(1)
        p1 = torch.sigmoid(gi / Tp)
        newv = (torch.rand(MHIS_CHAINS, device=dev) < p1).float()
        xprop = x.clone(); xprop.scatter_(1, i, newv[:, None])
        gp, gradp = grad_g(w, xprop)
        p1r = torch.sigmoid(gradp.gather(1, i).squeeze(1) / Tp)
        pf = torch.where(newv == 1, p1, 1 - p1)
        pr = torch.where(xi == 1, p1r, 1 - p1r)
        log_acc = (gp - gx) / T + torch.log(pr + 1e-12) - torch.log(pf + 1e-12)
        take = (torch.rand(MHIS_CHAINS, device=dev) < torch.exp(log_acc.clamp(max=0)))
        x = torch.where(take[:, None], xprop, x)
        if step >= MHIS_BURN:
            samples.append(x.clone())
    s = torch.cat(samples, 0)
    gs = g1(w, s); acc = (gs >= thr).float()
    est = (Z * (torch.exp((gmax - gs) / T) * acc).mean()).item()
    return est, s[acc.bool()]

In [7]:
# ---- run the sweep (both negative modes; bilinear first) ----
results = []
for neg_mode in NEG_MODES:
  for act in ACTS:
    for depth in DEPTHS:
        H = CONFIGS[(act, depth)]; ff = fwd_flops(depth, act, H)
        R1 = int(min(R_MAX, max(256, (SEARCH_FLOP_BUDGET / ff) / (2 * D * S_SWEEP * 4))))
        torch.manual_seed(0)
        org = Organisms(depth, act, H, B_ORGS, device)
        secrets = make_secrets(B_ORGS, device)
        print(f"[{neg_mode} | {act} depth{depth} H{H} ({n_params(depth,act,H)}p)] train {B_ORGS} | R1={R1}")
        t0 = time.time()
        loss = train(org, secrets, STEPS, device, neg_mode)
        mem_p, intruders, c99, c_thr, thr_vec, mdiff, bt = halo_brute(org, secrets, device)
        mem_min, mem_mean = mem_p.min().item(), mem_p.mean().item()
        mem_frac = (mem_p > 0.5).float().mean().item()
        true_p = c_thr / (1 << D)
        agg, flops = {}, []
        for b in range(B_ORGS):
            w = extract(org, b); FWD[0] = 0
            for k, v in battery(w, secrets[b], R1, device).items():
                agg[k] = agg.get(k, 0) + v / B_ORGS
            flops.append(FWD[0] * ff)
        sflops = sum(flops) / len(flops)
        est = {"gld": [], "itgis": [], "mhis": []}; rec = {"itgis": [], "mhis": []}
        for b in range(B_ORGS):
            w = extract(org, b); thr_b = thr_vec[b].item()
            est["gld"].append(gld(w, thr_b, device))
            e_i, ci = itgis(w, thr_b, device); est["itgis"].append(e_i)
            rec["itgis"].append(recov(ci, secrets[b]) if ci.numel() else 0)
            e_m, cm = mhis(w, thr_b, device);  est["mhis"].append(e_m)
            rec["mhis"].append(recov(cm, secrets[b]) if cm.numel() else 0)
        mean = lambda L: sum(L) / len(L)
        results.append(dict(neg_mode=neg_mode, act=act, depth=depth, H=H, params=n_params(depth, act, H),
                            loss=loss, mem_min=mem_min, mem_mean=mem_mean, mem_frac=mem_frac,
                            halo_intruders=intruders.mean().item(), halo_p99=c99.mean().item(),
                            consistency=mdiff, search_flops=sflops, **{f"srch_{k}": agg[k] for k in agg},
                            est_true=true_p.mean().item(), est_gld=mean(est["gld"]),
                            est_itgis=mean(est["itgis"]), est_mhis=mean(est["mhis"]),
                            rec_itgis=mean(rec["itgis"]), rec_mhis=mean(rec["mhis"])))
        lp = lambda v: (math.log10(v) if v > 0 else float('-inf'))
        print(f"   mem min {mem_min:.3f} mean {mem_mean:.3f} acc {100*mem_frac:.0f}% | "
              f"halo: {intruders.mean().item():.0f} >weakest-secret, {c99.mean().item():.0f} >p{HALO_P} "
              f"| consistency {mdiff:.1e} ({bt:.0f}s) | "
              f"search/16 " + " ".join(f"{k}:{agg[k]:.1f}" for k in agg)
              + f" | {sflops:.1e} flops ({sflops/READER_FLOPS:.0f}x reader)")
        print(f"   est log10 Pr[g>=weakest]: true {lp(true_p.mean().item()):.2f} | "
              f"GLD {lp(mean(est['gld'])):.2f} ITGIS {lp(mean(est['itgis'])):.2f} MHIS {lp(mean(est['mhis'])):.2f} "
              f"| IS rec/16 ITGIS {mean(rec['itgis']):.1f} MHIS {mean(rec['mhis']):.1f} | {time.time()-t0:.0f}s\n")
print("done.")

[natural | bilinear depth1 H176 (10413p)] train 4 | R1=1633
      step 5000/15000  loss 0.0000
      step 10000/15000  loss 0.0000
      step 15000/15000  loss 0.0000
   mem min 1.000 mean 1.000 acc 100% | halo: 4 >weakest-secret, 36 >p0.99 | consistency 0.0e+00 (38s) | search/16 rand_ICM:12.8 rand_2bit:11.5 grad_2bit:7.8 gcg:3.8 | 7.6e+10 flops (25x reader)
   est log10 Pr[g>=weakest]: true -7.13 | GLD -2.37 ITGIS -inf MHIS -23.03 | IS rec/16 ITGIS 0.0 MHIS 5.5 | 101s

[natural | bilinear depth2 H58 (10353p)] train 4 | R1=1628
      step 5000/15000  loss 0.0000
      step 10000/15000  loss 0.0000
      step 15000/15000  loss 0.0000
   mem min 1.000 mean 1.000 acc 100% | halo: 0 >weakest-secret, 16 >p0.99 | consistency 0.0e+00 (37s) | search/16 rand_ICM:0.0 rand_2bit:0.2 grad_2bit:0.0 gcg:0.0 | 7.6e+10 flops (25x reader)
   est log10 Pr[g>=weakest]: true -7.23 | GLD -5.55 ITGIS -inf MHIS -inf | IS rec/16 ITGIS 0.0 MHIS 0.0 | 117s

[natural | bilinear depth3 H43 (10220p)] train 4 | R1=1

In [8]:
# ---- results ----
import pandas as pd, math
df = pd.DataFrame(results)
srch_cols = [c for c in df.columns if c.startswith("srch_")]
df["srch_best"] = df[srch_cols].max(axis=1)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60)
print(df[["neg_mode","act","depth","H","params","loss","mem_frac","halo_intruders","halo_p99",
          "search_flops"] + srch_cols + ["srch_best"]].to_string(index=False))

def l10(v): return round(math.log10(v), 2) if v > 0 else float('-inf')
for c in ["est_true", "est_gld", "est_itgis", "est_mhis"]:
    df["log_" + c.split("_")[1]] = df[c].map(l10)

for nm in NEG_MODES:
    sub = df[df.neg_mode == nm]
    def pv(col): return sub.pivot(index="act", columns="depth", values=col)
    print(f"\n############### NEG_MODE = {nm} ###############")
    print("--- MEMORIZATION: frac of 16 secrets accepted (>0.5) ---");  print(pv("mem_frac").round(3).to_string())
    print("--- HALO: NON-secret inputs >= weakest secret (mean/org) ---"); print(pv("halo_intruders").round(0).to_string())
    print("--- SEARCH recovery /16, best over battery (lower=harder) ---"); print(pv("srch_best").round(2).to_string())
    print("--- ARC est log10 Pr[g>=weakest] (true vs estimators) + IS recovery/16 ---")
    print(sub[["act","depth","log_true","log_gld","log_itgis","log_mhis","rec_itgis","rec_mhis"]].to_string(index=False))

print(f"\nbrute/forward consistency max|diff| across configs: {df['consistency'].max():.2e}  (should be ~1e-5)")
print("natural (hard negs) -> small halo; random (no hard negs) -> large halo. QLD omitted (scalar logit -> naive).")

neg_mode      act  depth   H  params         loss  mem_frac  halo_intruders  halo_p99  search_flops  srch_rand_ICM  srch_rand_2bit  srch_grad_2bit  srch_gcg  srch_neuron  srch_neuron_hillclimb  srch_best
 natural bilinear      1 176   10413 5.083581e-07       1.0            3.75     35.50  7.564080e+10          12.75           11.50            7.75      3.75          NaN                    NaN      12.75
 natural bilinear      2  58   10353 3.601185e-08       1.0            0.00     16.00  7.562556e+10           0.00            0.25            0.00      0.00          NaN                    NaN       0.25
 natural bilinear      3  43   10220 1.635519e-07       1.0            0.00     16.75  7.545250e+10           0.25            0.50            0.00      0.00          NaN                    NaN       0.50
 natural   swiglu      1 176   10413 1.096638e-07       1.0            0.00     16.00  7.580808e+10          16.00           16.00           10.75     14.25          NaN               